In [1]:
import eurostat
from urllib.parse import urlparse, parse_qs
import pandas as pd

In [45]:
# 通过工具生成的url提取数据
def eurostat_get_standard_data_url(url):
    # 解析 URL
    parsed_url = urlparse(url)

    # 提取查询字符串
    query_str = parsed_url.query

    # 将查询字符串转换为字典
    params_dict = parse_qs(query_str)

    # 定位 'data/' 和 '?' 的位置
    start = url.find("data/") + len("data/")
    end = url.find("?", start)

    # 提取 'data/' 和 '?' 之间的内容
    database = url[start:end]
    # 获取数据
    Data = eurostat.get_data_df(database, filter_pars=params_dict,verbose=True)    

    # 获取label
    # 定位 'data/' 和 '?' 的位置
    start = url.find("data/") + len("data/")
    end = url.find("?", start)

    # 提取 'data/' 和 '?' 之间的内容
    database = url[start:end]

    # 获取有哪些被缩写了的属性
    pars = eurostat.get_pars(database)

    # 创建数据框，类名为pars加上对应的label
    pars_dict = dict()
    for par in pars:
        pars_dict[par] =  list(eurostat.get_dic(database, par,frmt="dict").keys())
        pars_dict[par+"_label"] = list(eurostat.get_dic(database, par,frmt="dict").values())

    # 找出最长的列表长度
    max_length = max(len(lst) for lst in pars_dict.values())

    # 用 None 或其他适当的值填充较短的列表
    for key in pars_dict:
        length = len(pars_dict[key])
        if length < max_length:
            pars_dict[key].extend([None] * (max_length - length))

    label = pd.DataFrame(pars_dict)

    # 给Data数据加上label
    Data_label = Data.merge(label[["geo","geo_label"]],left_on="geo\TIME_PERIOD",right_on="geo",how="left")
    for col in Data.columns:
        if col=="geo\TIME_PERIOD":
            break
        else:
            Data_label = Data_label.merge(label[[col,col+"_label"]],on=col,how="left")

    return Data_label

In [2]:
# 通过数据集名称database提取数据
def eurostat_get_standard_data_database(database):
    # 获取数据
    Data = eurostat.get_data_df(database,verbose=True)    

    # 获取label
    # 获取有哪些被缩写了的属性
    pars = eurostat.get_pars(database)

    # 创建数据框，类名为pars加上对应的label
    pars_dict = dict()
    for par in pars:
        pars_dict[par] =  list(eurostat.get_dic(database, par,frmt="dict").keys())
        pars_dict[par+"_label"] = list(eurostat.get_dic(database, par,frmt="dict").values())

    # 找出最长的列表长度
    max_length = max(len(lst) for lst in pars_dict.values())

    # 用 None 或其他适当的值填充较短的列表
    for key in pars_dict:
        length = len(pars_dict[key])
        if length < max_length:
            pars_dict[key].extend([None] * (max_length - length))

    label = pd.DataFrame(pars_dict)

    # 给Data数据加上label
    Data_label = Data.merge(label[["geo","geo_label"]],left_on="geo\TIME_PERIOD",right_on="geo",how="left")
    for col in Data.columns:
        if col=="geo\TIME_PERIOD":
            break
        else:
            Data_label = Data_label.merge(label[[col,col+"_label"]],on=col,how="left")

    return Data_label


In [48]:
D = eurostat_get_standard_data_url("https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/agr_r_accts?format=JSON&unit=MIO_EUR&unit=MIO_NAC&itm_newa=11000&itm_newa=11100&itm_newa=11200&itm_newa=11300&itm_newa=11400&itm_newa=11500&itm_newa=11900&itm_newa=12000&itm_newa=12200&itm_newa=12900&itm_newa=13000&indic_ag=PROD_BP&indic_ag=SUBS&indic_ag=TAX&indic_ag=PROD_PP&lang=en")

Download progress: 100.0%



In [ ]:
# 保存文件
col_order = ['freq', 'indic_ag', 'indic_ag_label','itm_newa','itm_newa_label' ,'unit','unit_label','geo\TIME_PERIOD','geo_label']
for year in range(1980,2022):
    # col_order = ['freq', 'animals', 'animals_label','unit','unit_label' ,'geo\TIME_PERIOD','geo_label',str(year)]
    Data_all_label_year = Data_all_label[col_order+[str(year)]]
    Data_all_label_year.to_csv("eurostat/anaimal_Economic_accounts_for_agriculture/"+str(year)+".csv",index=False,encoding='utf-8-sig')


In [47]:
D = eurostat_get_standard_data_database("agr_r_accts")

Download progress: 100.0%



In [7]:
Data = eurostat_get_standard_data_database("apro_ec_lshen")

Download progress: 100.0%



In [8]:
Data

,freq,animals,unit,geo\TIME_PERIOD,1960,1961,1962,1963,1964,1965,...,2008,2009,2010,2011,2012,geo,geo_label,freq_label,animals_label,unit_label
0,A,A5110O,THS_HD,AT,NaN,NaN,NaN,NaN,NaN,NaN,...,5918.9,5559.9,5724.5,NaN,NaN,AT,Austria,Annual,Laying hens,Thousand heads (animals)
1,A,A5110O,THS_HD,BE,15340.0,17333.0,16540.0,15590.0,15785.0,13736.0,...,8905.4,6483.7,NaN,NaN,NaN,BE,Belgium,Annual,Laying hens,Thousand heads (animals)
2,A,A5110O,THS_HD,BG,NaN,NaN,NaN,NaN,NaN,NaN,...,7116.5,6892.6,6217.0,NaN,NaN,BG,Bulgaria,Annual,Laying hens,Thousand heads (animals)
3,A,A5110O,THS_HD,CY,NaN,NaN,NaN,NaN,NaN,NaN,...,521.1,545.1,507.4,NaN,NaN,CY,Cyprus,Annual,Laying hens,Thousand heads (animals)
4,A,A5110O,THS_HD,CZ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,CZ,Czechia,Annual,Laying hens,Thousand heads (animals)
5,A,A5110O,THS_HD,DE,NaN,NaN,NaN,NaN,NaN,NaN,...,41323.0,36697.0,34036.0,NaN,NaN,DE,Germany,Annual,Laying hens,Thousand heads (animals)
6,A,A5110O,THS_HD,DK,NaN,NaN,NaN,NaN,NaN,6870.0,...,3521.0,3280.0,3900.0,NaN,NaN,DK,Denmark,Annual,Laying hens,Thousand heads (animals)
7,A,A5110O,THS_HD,EE,NaN,NaN,NaN,NaN,NaN,NaN,...,525.9,640.2,674.0,NaN,NaN,EE,Estonia,Annual,Laying hens,Thousand heads (animals)
8,A,A5110O,THS_HD,EL,NaN,NaN,NaN,NaN,NaN,NaN,...,12415.9,11983.6,11151.9,NaN,NaN,EL,Greece,Annual,Laying hens,Thousand heads (animals)
9,A,A5110O,THS_HD,ES,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,ES,Spain,Annual,Laying hens,Thousand heads (animals)


In [9]:
# 保存文件
col_order = ['freq', 'animals', 'animals_label','unit','unit_label' ,'geo\TIME_PERIOD','geo_label']
for year in range(1960,2013):
    # col_order = ['freq', 'animals', 'animals_label','unit','unit_label' ,'geo\TIME_PERIOD','geo_label',str(year)]
    Data_all_label_year = Data[col_order+[str(year)]]
    Data_all_label_year.to_csv("eurostat/animal_geo_ok_差禽/poultry/laying hens population/"+str(year)+".csv",index=False,encoding='utf-8-sig')
